# Kiểm Thử Thực Nghiệm Pass 1: Mô phỏng streaming ASR với Silero VAD & Zipformer-vi

**Đề tài:** Hệ thống phiên âm, tóm tắt và hỏi đáp nội dung cuộc họp  
**Tài liệu tham chiếu:** `docs/02-research-tech/asr-diarization-survey.md`  
**Môi trường thực thi:** Kaggle Notebook hoặc Google Colab  
**Lựa chọn cấu hình trên Kaggle:** **GPU T4** hoặc **CPU** (Zipformer-vi + Silero VAD siêu nhẹ, chạy mượt mà ngay cả trên CPU)  

---

### Mục tiêu và Cải Tiến Đột Phá Trong Phiên Bản Này
1. **Tích hợp Silero VAD Gatekeeper Nâng Cao:**
   - Ngưỡng VAD `0.55` là baseline cần đo lại theo bản ghi; không mặc định loại bỏ 100% nhạc nền hoặc tạp âm.
   - **Khoảng đệm âm thanh tối ưu (`speech_pad_ms = 120ms`):** Bảo toàn trọn vẹn âm tiết mở đầu ("Chào", "Dạ") mà không nuốt dính phần nhạc nền phía sau.
2. **Kiến Trúc mô phỏng streaming 2 Tầng (Two-tier Loop):**
   - **Tầng 1 (Partial Preview):** Cứ mỗi 450ms khi đang nói, hệ thống giải mã lại buffer bằng `OfflineRecognizer`; đây là mô phỏng partial, chưa phải streaming stateful.
   - **Tầng 2 (Boundary Finalization):** Khi người nói ngừng lời $\ge 500$ms hoặc câu chạm trần 6.5 giây, hệ thống chốt phân đoạn chính thức `[FINAL]`.
3. **Hậu xử lý văn bản & Từ điển Hotwords:** Tự động chuẩn hóa viết hoa đầu câu và sửa các từ viết tắt chuyên ngành/công nghệ phổ biến (`"ai ai"` $\to$ `"AI"`, `"kpi"` $\to$ `"KPI"`, `"ok"` $\to$ `"OK"`).
4. **Tương thích 100% Kaggle Mới Nhất:** Tận dụng PyTorch GPU CUDA có sẵn của Kaggle, không chạy lệnh `pip install torch` gây xung đột driver hoặc crash kernel.

## Bước 1: Cài đặt thư viện phụ thuộc (sherpa-onnx, torch, silero-vad, audio tools)

In [ ]:
# Lưu ý quan trọng cho Kaggle: Môi trường Kaggle đã tích hợp sẵn PyTorch GPU CUDA tối ưu.
# Tuyệt đối không cài đè torch/torchaudio vì sẽ gây xung đột driver CUDA hoặc lỗi kernel.
!pip install -q --no-warn-conflicts sherpa-onnx soundfile tabulate huggingface-hub silero-vad


## Bước 2: Tải Mô hình Tiếng Việt Zipformer-vi (sherpa-onnx 30M INT8)

Cơ chế tải kép tự động: Tải từ GitHub Release chính thức của k2-fsa, tự động fallback sang Hugging Face Hub nếu mạng gián đoạn.

In [ ]:
import os
import subprocess
from huggingface_hub import snapshot_download

MODEL_DIR = "sherpa-onnx-zipformer-vi-30M-int8-2026-02-09"

def setup_vietnamese_model():
    if os.path.exists(MODEL_DIR) and os.path.exists(os.path.join(MODEL_DIR, "tokens.txt")):
        print(f"Mô hình đã có sẵn trong thư mục: {MODEL_DIR}")
        return MODEL_DIR

    print("Đang tải mô hình Zipformer-vi tiếng Việt (dung lượng ~75 MB)...")
    archive_name = f"{MODEL_DIR}.tar.bz2"
    github_url = f"https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/{archive_name}"
    
    try:
        print(f"Thử tải từ GitHub: {github_url}")
        subprocess.run(["wget", "-q", "--tries=2", "--timeout=15", github_url], check=True)
        subprocess.run(["tar", "xjf", archive_name], check=True)
        if os.path.exists(MODEL_DIR):
            print("Tải và giải nén thành công từ GitHub!")
            return MODEL_DIR
    except Exception as e:
        print(f"Tải từ GitHub thất bại ({e}), chuyển sang tải từ Hugging Face Hub...")

    try:
        print("Đang tải từ Hugging Face Hub...")
        downloaded_dir = snapshot_download(
            repo_id="csukuangfj2/sherpa-onnx-zipformer-vi-30M-int8-2026-02-09",
            local_dir=MODEL_DIR
        )
        print("Tải thành công từ Hugging Face Hub!")
        return downloaded_dir
    except Exception as e2:
        print(f"Thử mirror Hugging Face: {e2}")
        return snapshot_download(
            repo_id="hoangmui410/sherpa-onnx-zipformer-vi-30M-int8",
            local_dir=MODEL_DIR
        )

active_model_dir = setup_vietnamese_model()
print(f"Thư mục mô hình hoạt động: {active_model_dir}")
!ls -lh {active_model_dir}

## Bước 3: Nạp Mô Hình Silero VAD (Voice Activity Detector) Phân Biệt Tiếng Nói & Nhạc Nền

In [ ]:
import torch

print("Đang nạp mô hình Silero VAD Gatekeeper...")
vad_model = None

# Phương án 1: Nạp trực tiếp qua package silero_vad (nhanh, ổn định trên Kaggle/Colab)
try:
    from silero_vad import load_silero_vad
    vad_model = load_silero_vad(onnx=False)
    print("Nạp Silero VAD thành công qua package silero_vad!")
except Exception as e:
    print(f"Nạp qua package silero_vad thất bại ({e}), chuyển sang thử PyTorch Hub...")

# Phương án 2: Fallback qua PyTorch Hub nếu cần
if vad_model is None:
    try:
        vad_model, _ = torch.hub.load(
            repo_or_dir="snakers4/silero-vad",
            model="silero_vad",
            force_reload=False,
            onnx=False,
            trust_repo=True
        )
        print("Nạp Silero VAD thành công từ PyTorch Hub!")
    except Exception as e2:
        raise RuntimeError(f"Không thể nạp Silero VAD: {e2}")

# Đặt VAD ở chế độ suy luận đánh giá
vad_model.eval()
print("Mô hình Silero VAD đã sẵn sàng làm chốt chặn Gatekeeper!")


## Bước 4: Tự Động Quét Nhận Diện Tệp Âm Thanh (.m4a, .mp3, .wav) & Chuẩn Hóa FFmpeg

In [ ]:
import glob
import subprocess
import soundfile as sf

def find_audio_file(explicit_path: str | None = None) -> str:
    """Resolve one input audio path without silently selecting a wrong recording.

    Args:
        explicit_path: User-provided path. If omitted, search common notebook folders.

    Returns:
        Path to the selected audio file.

    Raises:
        FileNotFoundError: If the explicit path does not exist or no audio is found.
        RuntimeError: If automatic discovery finds more than one candidate.
    """
    if explicit_path:
        if not os.path.isfile(explicit_path):
            raise FileNotFoundError(f"Không tìm thấy file âm thanh: {explicit_path}")
        return explicit_path
    extensions = ["*.m4a", "*.mp3", "*.wav", "*.aac", "*.flac", "*.ogg"]
    search_paths = [".", "/kaggle/input/**", "/kaggle/working/**"]
    
    found_files = []
    for sp in search_paths:
        for ext in extensions:
            matches = glob.glob(os.path.join(sp, ext), recursive=True)
            matches = [m for m in matches if "normalized" not in m and "live_input" not in m and "sherpa" not in m]
            found_files.extend(matches)
    
    found_files = sorted(set(found_files))
    if len(found_files) == 1:
        chosen = found_files[0]
        print(f"[TỰ ĐỘNG PHÁT HIỆN] Đã tìm thấy tệp âm thanh của bạn: {chosen}")
        return chosen
    if len(found_files) > 1:
        raise RuntimeError("Tìm thấy nhiều file âm thanh; hãy truyền INPUT_AUDIO_PATH rõ ràng: " + ", ".join(found_files))
    
    test_sample = f"{active_model_dir}/test_wavs/0.wav"
    if os.path.exists(test_sample):
        print(f"[MẪU CÓ SẴN] Sử dụng tệp âm thanh mẫu từ gói mô hình: {test_sample}")
        return test_sample
        
    print("[TẢI MẪU] Đang tải tệp âm thanh tiếng Việt mẫu về kiểm thử...")
    !wget -q -O meeting_sample.wav https://github.com/VinAIResearch/PhoWhisper/raw/main/assets/sample.wav
    return "meeting_sample.wav"

INPUT_AUDIO = find_audio_file(os.getenv("INPUT_AUDIO_PATH"))
NORMALIZED_AUDIO = "live_input_16k.wav"

# Chuẩn hóa về WAV Mono 16 kHz PCM 16-bit
cmd = [
    "ffmpeg", "-y", "-i", INPUT_AUDIO,
    "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le",
    NORMALIZED_AUDIO
]
subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)

audio_data, sample_rate = sf.read(NORMALIZED_AUDIO)
duration_sec = len(audio_data) / sample_rate
print(f"Đã chuẩn hóa thành công: {NORMALIZED_AUDIO}")
print(f"Thời lượng: {duration_sec:.2f} giây ({duration_sec/60:.2f} phút) | Tần số: {sample_rate} Hz")

## Bước 5: Khởi Tạo Bộ Nhận Dạng sherpa-onnx Zipformer-vi

In [ ]:
import sherpa_onnx

def get_onnx_file(dir_path, prefix):
    for fname in os.listdir(dir_path):
        if fname.startswith(prefix) and fname.endswith(".onnx"):
            return os.path.join(dir_path, fname)
    raise FileNotFoundError(f"Không tìm thấy tệp {prefix}*.onnx trong {dir_path}")

encoder_path = get_onnx_file(active_model_dir, "encoder")
decoder_path = get_onnx_file(active_model_dir, "decoder")
joiner_path = get_onnx_file(active_model_dir, "joiner")
tokens_path = os.path.join(active_model_dir, "tokens.txt")

print(f"Encoder: {os.path.basename(encoder_path)}")
print(f"Decoder: {os.path.basename(decoder_path)}")
print(f"Joiner:  {os.path.basename(joiner_path)}")
print(f"Tokens:  {os.path.basename(tokens_path)}")

recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(
    encoder=encoder_path,
    decoder=decoder_path,
    joiner=joiner_path,
    tokens=tokens_path,
    num_threads=4,
    sample_rate=16000,
    feature_dim=80,
    decoding_method="greedy_search"
)
print("Bộ nhận dạng sherpa-onnx đã sẵn sàng!")

## Bước 6: Mô Phỏng Luồng Trực Tiếp 2 Tầng (Offline Zipformer + VAD Boundary)

**Cơ chế hoạt động tối ưu:**
1. **Silero VAD Gatekeeper (`threshold = 0.55`):** Phân tích từng khung nhỏ 32ms. Đây là heuristic cần hiệu chỉnh theo bản ghi, không được xem là bộ lọc nhạc chính xác 100%.
2. **Tầng 1 (Interim / Partial Preview):** Cứ mỗi **450 mili-giây** khi người nói đang nói một câu dài, hệ thống giải mã lại buffer bằng `OfflineRecognizer`; đây là mô phỏng partial, chưa phải streaming stateful.
3. **Tầng 2 (Boundary Finalization):** Khi người nói ngừng lời $\ge 500$ms hoặc khi câu chạm trần 6.5 giây, hệ thống chốt câu `[FINAL]`, chạy bộ hậu chuẩn hóa viết hoa và thay thế từ khóa (`AI`, `KPI`, `OK`), lưu vào danh sách phân đoạn nháp.

In [ ]:
import time
import re
import numpy as np
import torch

def clean_vietnamese_text(text: str) -> str:
    """Normalize Zipformer text while preserving recognized content.

    Args:
        text: Raw decoder text.

    Returns:
        Normalized text with approved hotword casing.

    Raises:
        TypeError: If text is not a string.
    """
    if not isinstance(text, str):
        raise TypeError("text phải là chuỗi")
    t = text.strip()
    if not t:
        return ""
    t = t.lower()
    
    # Bảng thay thế âm học và thuật ngữ công nghệ / công sở phổ biến
    replacements = [
        (r"\bai ai\b", "AI"),
        (r"\ba i\b", "AI"),
        (r"\bkpi\b", "KPI"),
        (r"\bok\b", "OK"),
    ]
    for pattern, rep in replacements:
        t = re.sub(pattern, rep, t, flags=re.IGNORECASE)
    
    # Viết hoa ký tự đầu tiên
    return t[0].upper() + t[1:]

print("=== BẮT ĐẦU MÔ PHỎNG LUỒNG TRỰC TIẾP 2 TẦNG (PARTIAL & FINAL) ===\n")

# --- BỘ THAM SỐ THỰC NGHIỆM CHUẨN HÓA ---
VAD_FRAME_SIZE = 512            # 512 mẫu = 32ms ở tần số 16 kHz (chuẩn Silero VAD)
SAMPLE_RATE = 16000
VAD_THRESHOLD = 0.55            # Ngưỡng VAD tinh chỉnh (lọc sạch đoạn giao thoa nhạc nền)
SILENCE_THRESHOLD_SAMPLES = int(SAMPLE_RATE * 0.50)  # 500ms im lặng để chốt câu
PRE_PAD_SAMPLES = int(SAMPLE_RATE * 0.12)            # 120ms đệm trước để không nuốt âm đầu
MAX_SPEECH_SAMPLES = int(SAMPLE_RATE * 6.5)          # 6.5 giây trần phân đoạn tối đa
PARTIAL_INTERVAL_SAMPLES = int(SAMPLE_RATE * 0.45)   # Cứ 450ms cập nhật partial text một lần

live_draft_segments = []
ring_buffer = []  # Ring buffer lưu các mẫu trước đó để làm pre-padding

is_speaking = False
current_speech_samples = []
speech_start_sample = 0
silence_counter = 0
last_partial_sample = 0
segment_idx = 1

total_sim_start = time.time()
total_decode_time = 0.0

# Chuyển toàn bộ dữ liệu sang float32 chuẩn hóa
audio_float32 = audio_data.astype(np.float32)

for i in range(0, len(audio_float32) - VAD_FRAME_SIZE, VAD_FRAME_SIZE):
    frame = audio_float32[i : i + VAD_FRAME_SIZE]
    frame_tensor = torch.from_numpy(frame)
    
    # 1. Tính xác suất tiếng người bằng Silero VAD Gatekeeper
    with torch.no_grad():
        speech_prob = vad_model(frame_tensor, SAMPLE_RATE).item()
    
    # Duy trì ring buffer 120ms làm pre-padding
    ring_buffer.extend(frame)
    if len(ring_buffer) > PRE_PAD_SAMPLES:
        ring_buffer = ring_buffer[-PRE_PAD_SAMPLES:]
    
    # 2. Xử lý trạng thái tiếng người
    if speech_prob >= VAD_THRESHOLD:
        if not is_speaking:
            is_speaking = True
            speech_start_sample = max(0, i - len(ring_buffer))
            # ring_buffer đã chứa frame hiện tại; không append frame này lần nữa.
            current_speech_samples = list(ring_buffer)
            silence_counter = 0
            last_partial_sample = 0
        else:
            current_speech_samples.extend(frame)
        silence_counter = 0
        
        # --- TẦNG 1: PHÁT INTERIM / PARTIAL RESULTS (MỖI ~450MS) ---
        if (len(current_speech_samples) - last_partial_sample >= PARTIAL_INTERVAL_SAMPLES 
            and len(current_speech_samples) >= int(SAMPLE_RATE * 0.4)):
            stream_p = recognizer.create_stream()
            stream_p.accept_waveform(SAMPLE_RATE, np.array(current_speech_samples, dtype=np.float32))
            recognizer.decode_stream(stream_p)
            p_raw = stream_p.result.text.strip()
            if p_raw:
                p_clean = clean_vietnamese_text(p_raw)
                cur_s = (i / SAMPLE_RATE)
                print(f"  ... [PARTIAL {cur_s:4.1f}s] >> {p_clean}")
            last_partial_sample = len(current_speech_samples)

        # Cắt cưỡng bức khi câu liên tục vượt trần; trước đây chỉ kiểm tra trần trong nhánh silence.
        if len(current_speech_samples) >= MAX_SPEECH_SAMPLES:
            t_dec_start = time.time()
            stream_max = recognizer.create_stream()
            stream_max.accept_waveform(SAMPLE_RATE, np.array(current_speech_samples, dtype=np.float32))
            recognizer.decode_stream(stream_max)
            raw_text = stream_max.result.text.strip()
            total_decode_time += time.time() - t_dec_start
            if raw_text:
                cleaned_text = clean_vietnamese_text(raw_text)
                live_draft_segments.append({
                    "segment_id": f"draft_{segment_idx:04d}",
                    "start_ms": int(speech_start_sample / SAMPLE_RATE * 1000),
                    "end_ms": int((i + VAD_FRAME_SIZE) / SAMPLE_RATE * 1000),
                    "text": cleaned_text,
                    "source": "draft"
                })
                segment_idx += 1
            is_speaking = False
            current_speech_samples = []
            silence_counter = 0
            last_partial_sample = 0
            continue
            
    else:
        # Không có tiếng người (Nhạc nền hoặc Khoảng lặng)
        if is_speaking:
            current_speech_samples.extend(frame)
            silence_counter += VAD_FRAME_SIZE
            
            # --- TẦNG 2: CHỐT PHÂN ĐOẠN (BOUNDARY FINALIZATION) ---
            # Kích hoạt khi im lặng >= 500ms HOẶC thời lượng câu chạm trần 6.5 giây
            if silence_counter >= SILENCE_THRESHOLD_SAMPLES or len(current_speech_samples) >= MAX_SPEECH_SAMPLES:
                speech_end_sample = i - silence_counter
                duration_samples = len(current_speech_samples)
                
                # Bỏ qua nếu âm thanh quá ngắn (< 250ms)
                if duration_samples >= SAMPLE_RATE * 0.25:
                    t_dec_start = time.time()
                    
                    stream_f = recognizer.create_stream()
                    stream_f.accept_waveform(SAMPLE_RATE, np.array(current_speech_samples, dtype=np.float32))
                    recognizer.decode_stream(stream_f)
                    raw_text = stream_f.result.text.strip()
                    
                    t_dec_dur = (time.time() - t_dec_start) * 1000
                    total_decode_time += (t_dec_dur / 1000.0)
                    
                    if raw_text:
                        cleaned_text = clean_vietnamese_text(raw_text)
                        s_ms = int(speech_start_sample / SAMPLE_RATE * 1000)
                        e_ms = int(speech_end_sample / SAMPLE_RATE * 1000)
                        
                        print(f"[{s_ms/1000:5.1f}s -> {e_ms/1000:5.1f}s] [FINAL #{segment_idx:02d}] (Xử lý {t_dec_dur:4.1f}ms) >> {cleaned_text}\n")
                        
                        live_draft_segments.append({
                            "segment_id": f"draft_{segment_idx:04d}",
                            "start_ms": s_ms,
                            "end_ms": e_ms,
                            "text": cleaned_text,
                            "source": "draft"
                        })
                        segment_idx += 1
                
                is_speaking = False
                current_speech_samples = []
                silence_counter = 0
                last_partial_sample = 0

# Xử lý câu còn lại ở cuối tệp âm thanh (nếu có)
if is_speaking and len(current_speech_samples) >= SAMPLE_RATE * 0.25:
    stream_end = recognizer.create_stream()
    stream_end.accept_waveform(SAMPLE_RATE, np.array(current_speech_samples, dtype=np.float32))
    recognizer.decode_stream(stream_end)
    raw_text = stream_end.result.text.strip()
    if raw_text:
        cleaned_text = clean_vietnamese_text(raw_text)
        s_ms = int(speech_start_sample / SAMPLE_RATE * 1000)
        e_ms = int(len(audio_float32) / SAMPLE_RATE * 1000)
        print(f"[{s_ms/1000:5.1f}s -> {e_ms/1000:5.1f}s] [FINAL #{segment_idx:02d}] >> {cleaned_text}\n")
        live_draft_segments.append({
            "segment_id": f"draft_{segment_idx:04d}",
            "start_ms": s_ms,
            "end_ms": e_ms,
            "text": cleaned_text,
            "source": "draft"
        })

rtf_val = total_decode_time / max(duration_sec, 0.001)
print("=== KẾT QUẢ HIỆU NĂNG PASS 1 SAU KHI TỐI ƯU LIVE STREAMING 2 TẦNG ===")
print(f"Thời lượng âm thanh:          {duration_sec:.2f} giây")
print(f"Tổng thời gian decode ASR:    {total_decode_time:.2f} giây")
print(f"Hệ số thời gian thực (RTF):   {rtf_val:.4f} (Nhanh hơn thời gian thực {1/max(rtf_val, 0.0001):.1f} lần)")
print(f"Số phân đoạn draft chốt:      {len(live_draft_segments)}")


## Bước 7: Xuất Kết Quả JSON Phân Đoạn Nháp Đã Khử Ảo Giác

In [ ]:
import json
from tabulate import tabulate

print("\n--- BẢNG CÁC PHÂN ĐOẠN BẢN NHÁP (ĐÃ QUA SILERO VAD GATEKEEPER) ---")
table_rows = []
for s in live_draft_segments:
    table_rows.append([
        s["segment_id"],
        f"{s['start_ms']/1000:.1f}s - {s['end_ms']/1000:.1f}s",
        s["text"],
        s["source"]
    ])

if table_rows:
    print(tabulate(table_rows, headers=["Segment ID", "Khoảng thời gian", "Văn bản bản nháp", "Nguồn"], tablefmt="grid"))
else:
    print("Không có phát biểu nào được phát hiện (toàn bộ âm thanh là nhạc hoặc khoảng lặng).")

DRAFT_OUTPUT_JSON = "live_draft_segments_output.json"
with open(DRAFT_OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(live_draft_segments, f, ensure_ascii=False, indent=2)

print(f"\n[THÀNH CÔNG] Đã lưu tệp kết quả bản nháp: {DRAFT_OUTPUT_JSON}")

if live_draft_segments:
    print("\nMẫu cấu trúc phân đoạn nháp đầu tiên:")
    print(json.dumps(live_draft_segments[0], ensure_ascii=False, indent=2))